# Machote — Perceptrón
### Módulo IV · Tema 1.1 — Neuronas biológicas y el perceptrón

---
## ¿Qué es el Perceptrón y en qué se diferencia de la MPNeuron?

| | MPNeuron | Perceptrón |
|---|---|---|
| Entradas | Solo binarias (0 o 1) | Cualquier número |
| Pesos | Todos iguales (suma simple) | Cada entrada tiene su propio peso |
| Aprende | No — prueba thresholds por fuerza bruta | Sí — ajusta pesos con cada error |
| Bias | No tiene | Sí tiene (ajusta el umbral) |

**La diferencia clave:** el Perceptrón *aprende* ajustando sus pesos.
Cuando se equivoca, corrige un poco. Cuando acierta, no cambia nada.
Eso es el **algoritmo de aprendizaje del Perceptrón**.

## La fórmula matemática

```
z = (w₁·x₁) + (w₂·x₂) + ... + (wₙ·xₙ) + bias
  = Σ(wᵢ · xᵢ) + bias      ← esto es la sumatoria que ves en clase

salida = 1  si z >= 0
         0  si z <  0
```

Donde:
- **x** = las características de entrada (los datos)
- **w** = los pesos (weights) — lo que el modelo *aprende*
- **bias** = un ajuste extra, como una constante
- **z** = la suma ponderada (weighted sum)

## La regla de actualización (cómo aprende)

```
error = valor_real - valor_predicho

w_nuevo = w_viejo + (tasa_aprendizaje · error · x)
bias_nuevo = bias_viejo + (tasa_aprendizaje · error)
```

- Si **error = 0** (acertó) → los pesos no cambian
- Si **error = +1** (predijo 0, era 1) → aumenta los pesos
- Si **error = -1** (predijo 1, era 0) → disminuye los pesos

In [2]:
# ════════════════════════════════════════════════════════
# CELDA 1 — Imports y machote
# ════════════════════════════════════════════════════════
import sys, os

# Ruta relativa al machote (ajusta los '..' según tu estructura)
ruta_machote = os.path.abspath(os.path.join(os.getcwd(),'/home/usr-lbr-maq20/Documentos/Diplomado-RNA/Machote/')) #Machote\machote_ML.py D:\Proyectos\Diplomado-RNA\Machote\machote_ML.py
if ruta_machote not in sys.path:
    sys.path.append(ruta_machote)
from machote_ML import *

import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'seaborn'

---
## Implementación del Perceptrón desde cero

Implementarlo manualmente (sin sklearn) te ayuda a entender exactamente qué está pasando por dentro.

In [ ]:
# ════════════════════════════════════════════════════════
# CLASE: Perceptron (implementación manual)
# ════════════════════════════════════════════════════════
class Perceptron:
    def __init__(self, tasa_aprendizaje=0.01, epocas=100):
        """
        tasa_aprendizaje (learning rate):
            Qué tan grande es el ajuste de pesos en cada error.
            Muy grande → aprende rápido pero puede oscilar sin converger.
            Muy pequeña → aprende lento pero más estable.
            Valores típicos: 0.001, 0.01, 0.1

        epocas (epochs):
            Cuántas veces pasa por TODO el dataset durante el entrenamiento.
            1 época = ver cada muestra una vez.
            Más épocas = más oportunidades de corregir errores.
        """
        self.lr = tasa_aprendizaje
        self.epocas = epocas
        self.pesos = None   # se inicializan en fit()
        self.bias = None
        self.historial_errores = []  # para graficar cómo mejora

    def funcion_activacion(self, z):
        """
        Función escalón (step function).
        Convierte la suma ponderada z a 0 o 1.
        Si z >= 0 → 1 (activa la neurona)
        Si z <  0 → 0 (no activa)
        """
        return 1 if z >= 0 else 0

    def predic_uno(self, x):
        """
        Predice para UN solo ejemplo.
        x = array con las características de ese ejemplo.

        Proceso:
        1. Calcula z = suma de (peso * característica) + bias
           Esto es el producto punto (dot product): np.dot(pesos, x)
        2. Aplica la función de activación
        """
        z = np.dot(self.pesos, x) + self.bias
        return self.funcion_activacion(z)

    def predic(self, X):
        """Predice para TODOS los ejemplos en X."""
        return np.array([self.predic_uno(x) for x in X])

    def fit(self, X, Y):
        """
        Entrena el perceptrón.
        X → array de características (n_muestras, n_features)
        Y → array de etiquetas (0 o 1)

        PROCESO POR CADA ÉPOCA:
        1. Para cada muestra, predice
        2. Calcula el error = real - predicho
        3. Si hay error, ajusta los pesos y el bias
        4. Repite para todas las muestras
        """
        n_muestras, n_features = X.shape

        # Inicializar pesos en 0 (o pequeños números aleatorios)
        # Hay un peso por cada característica + un bias
        self.pesos = np.zeros(n_features)
        self.bias  = 0

        print(f"Entrenando {self.epocas} épocas con {n_muestras} muestras...")
        print(f"Tasa de aprendizaje: {self.lr}")
        print()

        for epoca in range(self.epocas):
            errores_epoca = 0

            for x, y_real in zip(X, Y):
                # Paso 1: predecir con los pesos actuales
                y_pred = self.predic_uno(x)

                # Paso 2: calcular el error
                error = y_real - y_pred
                # error = 0  → acertó, no hace nada
                # error = +1 → predijo 0, era 1 → aumenta pesos
                # error = -1 → predijo 1, era 0 → disminuye pesos

                # Paso 3: actualizar pesos solo si hubo error
                if error != 0:
                    # Regla de actualización del Perceptrón:
                    # w_nuevo = w_viejo + (lr * error * x)
                    self.pesos += self.lr * error * x
                    self.bias  += self.lr * error
                    errores_epoca += 1

            self.historial_errores.append(errores_epoca)

            # Mostrar progreso cada 10 épocas
            if (epoca + 1) % 10 == 0 or epoca == 0:
                acc = 1 - errores_epoca / n_muestras
                print(f"  Época {epoca+1:3d}/{self.epocas}: "
                      f"{errores_epoca} errores | accuracy aprox: {acc:.3f}")

            # Parar temprano si ya no hay errores (convergió)
            if errores_epoca == 0:
                print(f"\n¡Convergió en la época {epoca+1}! No hay más errores.")
                break

    def graficar_aprendizaje(self):
        """
        Grafica cómo disminuyen los errores con cada época.
        Una curva que baja hacia 0 = el modelo está aprendiendo bien.
        Una curva que no baja = los datos no son linealmente separables
        (el Perceptrón no puede separarlos con una línea recta).
        """
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(range(1, len(self.historial_errores) + 1),
                self.historial_errores, color='#D85A30', linewidth=2)
        ax.set_xlabel('Época')
        ax.set_ylabel('Número de errores')
        ax.set_title('Curva de aprendizaje del Perceptrón')
        ax.axhline(y=0, color='green', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

print("Clase Perceptron definida")

---
## Ejemplo 1 — Problema simple: compuerta AND

La compuerta AND es el ejemplo clásico para probar un Perceptrón:

| x1 | x2 | AND |
|----|----|-----|
| 0  | 0  |  0  |
| 0  | 1  |  0  |
| 1  | 0  |  0  |
| 1  | 1  |  1  |

Estos 4 puntos SÍ son linealmente separables — el Perceptrón puede aprenderlos.

In [ ]:
# ─── Datos de la compuerta AND ───────────────────────────────────────
X_and = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])
Y_and = np.array([0, 0, 0, 1])

# ─── Entrenar ────────────────────────────────────────────────────────
p = Perceptron(tasa_aprendizaje=0.1, epocas=20)
p.fit(X_and, Y_and)

# ─── Evaluar ─────────────────────────────────────────────────────────
print("\nPredicciones finales:")
for x, y_real in zip(X_and, Y_and):
    y_pred = p.predic_uno(x)
    correcto = "✅" if y_pred == y_real else "❌"
    print(f"  AND({x[0]}, {x[1]}) = {y_pred}  (real: {y_real}) {correcto}")

print(f"\nPesos aprendidos: {p.pesos}")
print(f"Bias aprendido:   {p.bias}")
p.graficar_aprendizaje()

---
## Ejemplo 2 — Dataset real: Breast Cancer

Ahora usamos el mismo dataset del diplomado pero con el Perceptrón en lugar de la MPNeuron.
Nótese que **no necesitamos binarizar** — el Perceptrón acepta valores continuos directamente.

In [ ]:
from sklearn.datasets import load_breast_cancer

# Cargar y preparar datos
X, Y, clases = cargar_desde_sklearn(load_breast_cancer)
correlaciones = ver_correlacion_con_y(X, Y, umbral=0.5)
X_red, features = seleccionar_features(X, Y, umbral=0.5)
X_train, X_test, y_train, y_test = dividir_datos(X_red, Y)

# IMPORTANTE: el Perceptrón es muy sensible a la escala
# Siempre escalar antes de usar
X_train_sc, X_test_sc = escalar_datos(X_train, X_test)

In [ ]:
# Entrenar el Perceptrón
perc = Perceptron(tasa_aprendizaje=0.01, epocas=100)
perc.fit(X_train_sc, y_train)

In [ ]:
# Evaluar y comparar con la MPNeuron
Y_pred = perc.predic(X_test_sc)
acc_perc = evaluar_clasificacion(y_test, Y_pred, clases)
perc.graficar_aprendizaje()

# Comparar modelos
comparar_modelos({
    'MPNeuron (clase)': 0.916,   # resultado de la clase anterior
    'Perceptrón': acc_perc
})

---
## Ejemplo 3 — Perceptrón de sklearn

En la práctica nadie implementa el Perceptrón desde cero.
sklearn tiene una implementación optimizada que hace lo mismo pero más rápido.

In [ ]:
from sklearn.linear_model import Perceptron as SklearnPerceptron

# ¿Cómo se diferencia de nuestra implementación?
# - max_iter = épocas
# - eta0     = tasa de aprendizaje
# - tol      = si el error baja menos de este valor entre épocas, para automático
# - random_state = semilla para reproducibilidad

modelo_sk = SklearnPerceptron(
    max_iter=100,
    eta0=0.01,
    tol=1e-3,
    random_state=42
)
modelo_sk.fit(X_train_sc, y_train)
Y_pred_sk = modelo_sk.predict(X_test_sc)

print("Perceptrón de sklearn:")
acc_sk = evaluar_clasificacion(y_test, Y_pred_sk, clases)

print(f"\nPesos aprendidos: {modelo_sk.coef_[0][:5]}... ({len(modelo_sk.coef_[0])} en total)")
print(f"Bias aprendido:   {modelo_sk.intercept_[0]:.4f}")

---
## ¿Qué sigue después del Perceptrón?

El Perceptrón tiene una limitación importante: **solo puede separar datos linealmente separables**.

Ejemplo clásico de la compuerta XOR — el Perceptrón NO puede aprenderla:

| x1 | x2 | XOR |
|----|----|-----|
| 0  | 0  |  0  |
| 0  | 1  |  1  |
| 1  | 0  |  1  |
| 1  | 1  |  0  |

No puedes separar estos 4 puntos con una sola línea recta.

**La solución:** apilar varias capas de Perceptrones → **Perceptrón Multicapa (MLP)**.
Eso es lo que veremos en el Módulo IV: redes neuronales profundas.